# NetraSetu - Model 1: DR Severity Classifier (Branch A)

Self-contained training notebook for **Kaggle** (free T4 GPU).

## Before you run
1. Settings -> Accelerator -> **GPU T4 x1** (P100 also fine).
2. Settings -> Internet -> **On** (needed once, to download EfficientNet-B0 ImageNet weights).
3. Add data:
   - Competitions -> **APTOS 2019 Blindness Detection** -> Add. Mounts at `/kaggle/input/aptos2019-blindness-detection/`.
   - Your Datasets -> add the **IDRiD Disease Grading** dataset you uploaded privately, then set `IDRID_ROOT` in the config cell to its mount path (`/kaggle/input/<your-slug>`).

## What it does
- Inlines the Ben Graham preprocessing from `preprocessing/ben_graham.py` (circular crop -> resize 384 -> Gaussian-subtraction contrast boost), caches results to disk so it is not redone every epoch.
- APTOS (`diagnosis`) + IDRiD (`Retinopathy grade`) -> one 0-4 label space.
- 70/15/15 stratified split computed **separately per source**, then merged, so IDRiD is not diluted by the larger APTOS pool.
- EfficientNet-B0 (`timm`, ImageNet-pretrained, 5-class head, dropout kept before the head for later MC-Dropout).
- Ordinal-aware weighted cross-entropy: `CE * (1 + |pred_grade - true_grade|)` with inverse-frequency class weights `total / (5 * count_c)`.
- AdamW (lr 1e-4, wd 1e-5), cosine annealing, batch 32, up to 30 epochs, early stop on val quadratic-weighted kappa (patience 7).
- Per epoch: QWK, macro-F1, referable-DR (grade >= 2) sensitivity / specificity on the val split.
- Saves `branchA_v1.pt` + validation logits/labels `.npy` for temperature scaling (Prompt 6a).

All seeds fixed. Targets: val/test QWK > 0.85, referable sensitivity > 0.90, specificity > 0.85.

In [ ]:
import os

# =====================================================================
#  CONFIG  -  edit IDRID_ROOT after adding your private IDRiD dataset
# =====================================================================

# >>> EDIT THIS <<<  mount path of your uploaded IDRiD "Disease Grading" dataset
# e.g. "/kaggle/input/idrid-disease-grading"
IDRID_ROOT = "/kaggle/input/idrid-disease-grading"

# APTOS 2019 competition data (standard Kaggle mount - normally unchanged)
APTOS_ROOT = "/kaggle/input/aptos2019-blindness-detection"

# Toggles
USE_IDRID = True               # False -> train on APTOS only (quick pipeline check)
DEBUG_MAX_PER_SOURCE = None     # e.g. 200 -> cap images/source for a fast end-to-end test

# Paths
OUT_DIR = "/kaggle/working"                 # downloadable after the run
CACHE_DIR = "/kaggle/temp/bg_cache_384"     # ephemeral scratch, not saved as output

# Hyperparameters (training plan - Model 1)
SEED = 42
IMG_SIZE = 384
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 7
LR = 1e-4
WEIGHT_DECAY = 1e-5
DROP_RATE = 0.3                 # dropout before the classifier head (MC-Dropout later)
NUM_WORKERS = 4
USE_AMP = True
MODEL_NAME = "efficientnet_b0"  # fallbacks: "resnet50", or "tf_efficientnet_b0" on old timm
NUM_CLASSES = 5

CKPT_PATH        = f"{OUT_DIR}/branchA_v1.pt"
VAL_LOGITS_PATH  = f"{OUT_DIR}/branchA_v1_val_logits.npy"
VAL_LABELS_PATH  = f"{OUT_DIR}/branchA_v1_val_labels.npy"
VAL_IDS_PATH     = f"{OUT_DIR}/branchA_v1_val_ids.npy"
TEST_LOGITS_PATH = f"{OUT_DIR}/branchA_v1_test_logits.npy"
TEST_LABELS_PATH = f"{OUT_DIR}/branchA_v1_test_labels.npy"
TEST_IDS_PATH    = f"{OUT_DIR}/branchA_v1_test_ids.npy"
METRICS_PATH     = f"{OUT_DIR}/branchA_v1_metrics.json"

print("IDRID_ROOT:", IDRID_ROOT, "| exists:", os.path.isdir(IDRID_ROOT))
print("APTOS_ROOT:", APTOS_ROOT, "| exists:", os.path.isdir(APTOS_ROOT))

In [ ]:
import os, json, random, time, glob, warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| timm", timm.__version__, "| device:", device)
if device.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0), f"| {p.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected - enable Settings -> Accelerator -> GPU T4")

# ---- AMP compat (torch.amp on new torch, torch.cuda.amp on old) ----
try:
    from torch.amp import autocast as _ac, GradScaler as _GS
    def amp_autocast():
        return _ac("cuda", enabled=USE_AMP and device.type == "cuda")
    def make_scaler():
        return _GS("cuda", enabled=USE_AMP and device.type == "cuda")
except Exception:
    from torch.cuda.amp import autocast as _ac, GradScaler as _GS
    def amp_autocast():
        return _ac(enabled=USE_AMP and device.type == "cuda")
    def make_scaler():
        return _GS(enabled=USE_AMP and device.type == "cuda")

# ---- reproducibility ----
def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    s = torch.initial_seed() % (2 ** 32)
    np.random.seed(s)
    random.seed(s)

seed_everything(SEED)
g = torch.Generator()
g.manual_seed(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("cache dir :", CACHE_DIR)
print("output dir:", OUT_DIR)

## 1. Ben Graham preprocessing (inlined)

Identical algorithm to `preprocessing/ben_graham.py`:
1. Green-channel threshold (`> 7`) -> morphological close -> largest external contour -> crop to its bounding box (removes the black surround).
2. Resize to `IMG_SIZE x IMG_SIZE` with `INTER_AREA`.
3. Local-contrast boost: `clip(4*img - 4*GaussianBlur(img) + 128)`, `sigma = IMG_SIZE/30`, `ksize = 2*floor(sigma) + 1`.

Images are read BGR (`cv2.imread`); the maths are channel-order independent (green is index 1 in both BGR and RGB). The only addition here is a `BGR -> RGB` swap **after** Ben Graham (done in the caching step) because the ImageNet-pretrained backbone expects RGB.

In [ ]:
def ben_graham_preprocess(image: np.ndarray, target_size: int = 384) -> np.ndarray:
    """
    Ben Graham-style fundus preprocessing. Identical to preprocessing/ben_graham.py.

    image       : BGR uint8 array, shape (H, W, 3)
    target_size : output square side length
    returns     : uint8 array (target_size, target_size, 3), values 0..255, BGR order
    """
    if image is None or image.size == 0:
        raise ValueError("ben_graham_preprocess received an empty or None image.")
    if image.ndim != 3 or image.shape[2] != 3:
        raise ValueError(f"Expected a 3-channel image, got shape {image.shape}.")

    # Step 1: detect the retinal circle and crop to it
    gray = image[:, :, 1]  # green channel
    _, mask = cv2.threshold(gray, 7, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        x = max(0, x)
        y = max(0, y)
        w = min(w, image.shape[1] - x)
        h = min(h, image.shape[0] - y)
        cropped = image[y:y + h, x:x + w]
    else:
        cropped = image
    if cropped.size == 0:
        cropped = image

    # Step 2: resize
    resized = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)

    # Step 3: local contrast via Gaussian subtraction
    sigma = target_size / 30.0
    ksize = int(sigma) * 2 + 1
    ksize = max(ksize, 1)
    blurred = cv2.GaussianBlur(resized, (ksize, ksize), sigma)
    enhanced = cv2.addWeighted(resized, 4, blurred, -4, 128)
    return enhanced


# quick self-test on a synthetic disc
_t = np.zeros((512, 512, 3), np.uint8)
cv2.circle(_t, (256, 256), 200, (40, 180, 60), -1)
_o = ben_graham_preprocess(_t, IMG_SIZE)
print("self-test OK - output", _o.shape, _o.dtype, "range", int(_o.min()), int(_o.max()))

## 2. Build the combined label table

APTOS `train.csv` (`id_code`, `diagnosis`) + every IDRiD "Retinopathy grade" CSV found anywhere under `IDRID_ROOT`. IDRiD rows are matched to image files by name **within their own train/test folder** and tagged, so train/test images that share a number never collide.

In [ ]:
def load_aptos(root):
    csv = os.path.join(root, "train.csv")
    img_dir = os.path.join(root, "train_images")
    if not os.path.isfile(csv):
        raise FileNotFoundError(
            f"APTOS train.csv not found at {csv}. Add the 'APTOS 2019 Blindness "
            f"Detection' competition data to this notebook.")
    df = pd.read_csv(csv).rename(columns={"id_code": "image_id", "diagnosis": "grade"})
    df["path"] = df["image_id"].apply(lambda s: os.path.join(img_dir, f"{s}.png"))
    df["source"] = "aptos"
    df["grade"] = pd.to_numeric(df["grade"], errors="coerce")
    df = df.dropna(subset=["grade"])
    df["grade"] = df["grade"].astype(int)
    return df[["image_id", "path", "grade", "source"]]


def _tag_of(path):
    low = path.lower()
    if "test" in low:
        return "test"
    if "train" in low:
        return "train"
    return "all"


def load_idrid(root):
    if not os.path.isdir(root):
        raise FileNotFoundError(
            f"IDRID_ROOT '{root}' does not exist. Upload IDRiD's Disease Grading subset "
            f"as a private Kaggle dataset, add it here, and set IDRID_ROOT to its "
            f"/kaggle/input/... path.")

    imgs = []
    for e in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.png"):
        imgs += glob.glob(os.path.join(root, "**", e), recursive=True)
    by_tag_stem, by_stem = {}, {}
    for p in imgs:
        stem = os.path.splitext(os.path.basename(p))[0]
        by_tag_stem.setdefault((_tag_of(p), stem), p)
        by_stem.setdefault(stem, p)

    rows = []
    for c in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        try:
            t = pd.read_csv(c)
        except Exception:
            continue
        t.columns = [str(x).strip() for x in t.columns]
        gcol = next((col for col in t.columns
                     if col.lower().replace("_", " ").startswith("retinopathy grade")), None)
        ncol = next((col for col in t.columns
                     if col.lower().replace("_", " ").startswith("image name")), None)
        if not (gcol and ncol):
            continue
        ctag = _tag_of(c)
        t = t[[ncol, gcol]].rename(columns={ncol: "name", gcol: "grade"})
        t["name"] = t["name"].astype(str).str.strip()
        t["grade"] = pd.to_numeric(t["grade"], errors="coerce")
        t = t.dropna(subset=["name", "grade"])
        n_ok = 0
        for name, grade in zip(t["name"], t["grade"]):
            p = by_tag_stem.get((ctag, name)) or by_stem.get(name)
            if p is None:
                continue
            rows.append({"image_id": f"idrid_{ctag}_{name}", "path": p,
                         "grade": int(grade), "source": "idrid"})
            n_ok += 1
        print(f"  IDRiD CSV: {os.path.relpath(c, root)}  tag={ctag}  matched {n_ok}/{len(t)}")
    if not rows:
        raise FileNotFoundError(
            f"No IDRiD 'Retinopathy grade' CSV rows could be matched to image files "
            f"under {root}. Expected a CSV with 'Image name' + 'Retinopathy grade' columns.")
    df = pd.DataFrame(rows).drop_duplicates(subset="image_id").reset_index(drop=True)
    return df[["image_id", "path", "grade", "source"]]


frames = [load_aptos(APTOS_ROOT)]
if USE_IDRID:
    frames.append(load_idrid(IDRID_ROOT))
else:
    print("USE_IDRID = False -> APTOS only")

data = pd.concat(frames, ignore_index=True)
data = data[data["grade"].between(0, 4)].reset_index(drop=True)

exists = data["path"].apply(os.path.isfile)
if (~exists).any():
    print(f"WARNING: {(~exists).sum()} rows point to a missing image file - dropped")
data = data[exists].reset_index(drop=True)

if DEBUG_MAX_PER_SOURCE:
    data = pd.concat([
        d.sample(min(len(d), DEBUG_MAX_PER_SOURCE), random_state=SEED)
        for _, d in data.groupby("source")
    ]).reset_index(drop=True)
    print(f"DEBUG_MAX_PER_SOURCE={DEBUG_MAX_PER_SOURCE} -> {len(data)} images")

print("\nTotal usable images:", len(data))
print(pd.crosstab(data["source"], data["grade"], margins=True))

## 3. Stratified 70/15/15 split, per source, then merge

`train_test_split` is called twice per source (70 / 30, then 15 / 15 of the remainder), stratified by grade each time. The same-role halves from APTOS and IDRiD are then concatenated. A leakage assert confirms no image id appears in more than one split.

In [ ]:
def split_70_15_15(df, seed=SEED):
    idx = np.arange(len(df))
    y = df["grade"].values
    tr, tmp = train_test_split(idx, test_size=0.30, random_state=seed, stratify=y)
    va, te = train_test_split(tmp, test_size=0.50, random_state=seed, stratify=y[tmp])
    return df.iloc[tr], df.iloc[va], df.iloc[te]


parts = {"train": [], "val": [], "test": []}
for src, d in data.groupby("source"):
    d = d.reset_index(drop=True)
    tr, va, te = split_70_15_15(d, SEED)
    parts["train"].append(tr)
    parts["val"].append(va)
    parts["test"].append(te)
    print(f"{src:6s} -> train {len(tr):4d} | val {len(va):3d} | test {len(te):3d}")

train_df = pd.concat(parts["train"]).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(parts["val"]).reset_index(drop=True)
test_df = pd.concat(parts["test"]).reset_index(drop=True)

a = set(train_df.image_id)
b = set(val_df.image_id)
c = set(test_df.image_id)
assert not (a & b) and not (a & c) and not (b & c), "split leakage detected"

print(f"\nCOMBINED  train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
print("\ntrain grade x source:")
print(pd.crosstab(train_df.source, train_df.grade, margins=True))
print("\nval  grade dist:", val_df.grade.value_counts().sort_index().to_dict())
print("test grade dist:", test_df.grade.value_counts().sort_index().to_dict())

## 4. Ben Graham preprocess + cache to disk

Every image is preprocessed once and cached as a `.npy` (RGB uint8) under `CACHE_DIR` (ephemeral Kaggle scratch, not saved as output). Re-running this cell is cheap - existing caches are skipped. Images that fail to read/preprocess are reported and dropped from the splits.

In [ ]:
def cache_key(source, image_id):
    return f"{source}__{image_id}"

def cache_path_for(source, image_id):
    return os.path.join(CACHE_DIR, cache_key(source, image_id) + ".npy")

def preprocess_one(args):
    source, image_id, path = args
    cp = cache_path_for(source, image_id)
    if os.path.isfile(cp):
        return (source, image_id, True, "cached")
    try:
        img = cv2.imread(path, cv2.IMREAD_COLOR)          # BGR
        if img is None:
            return (source, image_id, False, "unreadable")
        proc = ben_graham_preprocess(img, IMG_SIZE)       # BGR, IMG_SIZE square
        proc = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)      # -> RGB for the pretrained net
        np.save(cp, proc)
        return (source, image_id, True, "ok")
    except Exception as e:
        return (source, image_id, False, repr(e))


all_rows = pd.concat([train_df, val_df, test_df])[["source", "image_id", "path"]]
jobs = list(all_rows.itertuples(index=False, name=None))

print(f"Preprocessing {len(jobs)} images (Ben Graham {IMG_SIZE}px) -> {CACHE_DIR}")
t0 = time.time()
fails = []
with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as ex:
    for source, image_id, ok, msg in tqdm(ex.map(preprocess_one, jobs),
                                          total=len(jobs), mininterval=5.0):
        if not ok:
            fails.append((source, image_id, msg))
print(f"done in {time.time() - t0:.0f}s | failures: {len(fails)}")
for f in fails[:20]:
    print("  FAIL", f)

if fails:
    bad = {(s, i) for s, i, _ in fails}
    for nm in ("train_df", "val_df", "test_df"):
        d = globals()[nm]
        keep = ~d.apply(lambda r: (r.source, r.image_id) in bad, axis=1)
        globals()[nm] = d[keep].reset_index(drop=True)
    print(f"after dropping failures -> train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

## 5. Dataset + augmentation

Train augmentation: random zoom/crop (`RandomResizedCrop`, scale 0.8-1.0), rotation +/-20 deg, horizontal flip, brightness/contrast jitter +/-15%. Eval: resize + ImageNet normalize only. Cached arrays are already RGB uint8 384px.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1111)),
    T.RandomRotation(20, fill=0),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class RetinaDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        arr = np.load(cache_path_for(r.source, r.image_id))   # RGB uint8 HWC
        img = self.transform(Image.fromarray(arr))
        return img, int(r.grade), cache_key(r.source, r.image_id)


train_ds = RetinaDataset(train_df, train_tf)
val_ds = RetinaDataset(val_df, eval_tf)
test_ds = RetinaDataset(test_df, eval_tf)

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
                 worker_init_fn=seed_worker, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, generator=g, **loader_kw)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kw)
test_loader = DataLoader(test_ds, shuffle=False, **loader_kw)

xb, yb, idb = next(iter(train_loader))
print("sample batch:", tuple(xb.shape), xb.dtype, "| labels", yb[:8].tolist())

## 6. Model - EfficientNet-B0 (timm, ImageNet-pretrained)

A small `DRClassifier` wrapper runs the timm backbone in feature mode (`num_classes=0`) and adds an **explicit `nn.Dropout` + fresh `nn.Linear(->5)` head**. The explicit module matters: timm's built-in head dropout is functional (`F.dropout`) with no module for Prompt 6a's MC-Dropout to re-enable at inference. Re-run this cell to reset weights before re-training.

In [ ]:
class DRClassifier(nn.Module):
    """timm backbone (feature mode) + explicit Dropout + Linear head.

    The explicit nn.Dropout before the head is what Prompt 6a's MC-Dropout toggles
    at inference; timm's own head dropout is a functional call with no module.
    """
    def __init__(self, model_name, num_classes, drop_rate, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                          num_classes=0, drop_rate=0.0)
        self.num_features = self.backbone.num_features
        self.drop = nn.Dropout(p=drop_rate)
        self.head = nn.Linear(self.num_features, num_classes)

    def forward(self, x):
        return self.head(self.drop(self.backbone(x)))


ARCH_DESC = ("DRClassifier: timm(model_name, num_classes=0, drop_rate=0) -> "
             "nn.Dropout(drop_rate) -> nn.Linear(num_features, num_classes)")

try:
    model = DRClassifier(MODEL_NAME, NUM_CLASSES, DROP_RATE, pretrained=True).to(device)
except Exception as e:
    raise RuntimeError(
        "Could not create the pretrained model. Turn Settings -> Internet -> On so timm "
        "can download ImageNet weights, then re-run this cell.") from e

n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_drop = sum(1 for m in model.modules() if isinstance(m, nn.Dropout) and m.p > 0)
print(f"{MODEL_NAME}: {n_params:.1f}M params | head = Dropout(p={DROP_RATE}) + "
      f"Linear({model.num_features} -> {NUM_CLASSES}) | nn.Dropout modules: {n_drop}")
assert n_drop >= 1, "expected a real nn.Dropout before the head for MC-Dropout"

## 7. Loss - ordinal-aware weighted cross-entropy

`weight_c = total / (5 * count_c)` from the **training** split grade distribution. Per sample: `CE_weighted * (1 + |argmax(logits) - true_grade|)`, then mean over the batch.

In [ ]:
counts = (train_df["grade"].value_counts()
          .reindex(range(NUM_CLASSES), fill_value=0).sort_index())
total = int(counts.sum())
w = total / (NUM_CLASSES * counts.clip(lower=1).astype(float))   # total / (5 * count_c)
class_weights = torch.tensor(w.values, dtype=torch.float32, device=device)
print("train grade counts:", counts.to_dict())
print("class weights     :", {k: round(v, 3) for k, v in w.to_dict().items()})


class OrdinalWeightedCE(nn.Module):
    """Inverse-freq weighted CE, scaled per sample by (1 + |argmax(logits) - target|)."""
    def __init__(self, class_weights):
        super().__init__()
        self.register_buffer("cw", class_weights.detach().clone())

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.cw, reduction="none")
        pred = logits.detach().argmax(dim=1)
        factor = 1.0 + (pred - target).abs().float()
        return (ce * factor).mean()


criterion = OrdinalWeightedCE(class_weights).to(device)

## 8. Metrics

Quadratic-weighted kappa, macro F1, and referable-DR (grade >= 2) sensitivity / specificity. `evaluate` also returns raw logits + labels + ids so the best epoch's validation logits can be saved for temperature scaling.

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mf1 = f1_score(y_true, y_pred, average="macro")
    t = y_true >= 2
    p = y_pred >= 2
    tp = int(np.sum(p & t)); fn = int(np.sum(~p & t))
    tn = int(np.sum(~p & ~t)); fp = int(np.sum(p & ~t))
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    return dict(qwk=qwk, macro_f1=mf1, ref_sens=sens, ref_spec=spec,
                ref_tp=tp, ref_fn=fn, ref_tn=tn, ref_fp=fp)


@torch.no_grad()
def evaluate(model, loader, criterion=None):
    model.eval()
    logits_all, labels_all, ids_all = [], [], []
    loss_sum, n = 0.0, 0
    for imgs, labels, ids in loader:
        imgs = imgs.to(device, non_blocking=True)
        with amp_autocast():
            logits = model(imgs)
            if criterion is not None:
                loss = criterion(logits, labels.to(device, non_blocking=True))
                loss_sum += loss.item() * imgs.size(0)
                n += imgs.size(0)
        logits_all.append(logits.float().cpu().numpy())
        labels_all.append(labels.numpy())
        ids_all.extend(list(ids))
    logits = np.concatenate(logits_all)
    labels = np.concatenate(labels_all)
    m = compute_metrics(labels, logits.argmax(1))
    if criterion is not None:
        m["loss"] = loss_sum / max(n, 1)
    return m, logits, labels, np.array(ids_all)

## 9. Train

AdamW (lr 1e-4, wd 1e-5) + cosine annealing over `EPOCHS`, batch 32, mixed precision. Early stopping on validation QWK, patience 7. The best checkpoint and its validation logits/labels are written every time QWK improves, so a killed session still leaves the best model on disk.

In [ ]:
seed_everything(SEED)   # reproducible training trajectory
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = make_scaler()

best_qwk, best_epoch, no_improve = -1.0, -1, 0
history = []

print(f"train {len(train_df)} | val {len(val_df)} | batch {BATCH_SIZE} | "
      f"up to {EPOCHS} epochs | early stop patience {PATIENCE}\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, seen, t0 = 0.0, 0, time.time()
    pbar = tqdm(train_loader, desc=f"epoch {epoch:02d}/{EPOCHS} [train]",
                leave=False, mininterval=5.0)
    for imgs, labels, _ in pbar:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with amp_autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item() * imgs.size(0)
        seen += imgs.size(0)
        pbar.set_postfix(loss=f"{run_loss / seen:.4f}")
    scheduler.step()
    train_loss = run_loss / max(seen, 1)

    vm, v_logits, v_labels, v_ids = evaluate(model, val_loader, criterion)
    lr_now = optimizer.param_groups[0]["lr"]
    dt = time.time() - t0
    improved = vm["qwk"] > best_qwk + 1e-5
    flag = "   <-- new best" if improved else ""
    print(f"epoch {epoch:02d}/{EPOCHS} | {dt:5.0f}s | lr {lr_now:.2e} | "
          f"train_loss {train_loss:.4f} | val_loss {vm['loss']:.4f} | "
          f"QWK {vm['qwk']:.4f} | macroF1 {vm['macro_f1']:.4f} | "
          f"refDR sens {vm['ref_sens']:.3f} spec {vm['ref_spec']:.3f}{flag}")

    history.append({
        "epoch": epoch, "train_loss": float(train_loss), "lr": float(lr_now),
        "seconds": float(dt), "val_loss": float(vm["loss"]),
        "val_qwk": float(vm["qwk"]), "val_macro_f1": float(vm["macro_f1"]),
        "val_ref_sens": float(vm["ref_sens"]), "val_ref_spec": float(vm["ref_spec"]),
    })

    if improved:
        best_qwk, best_epoch, no_improve = vm["qwk"], epoch, 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "arch": ARCH_DESC,
            "num_classes": NUM_CLASSES,
            "num_features": int(model.num_features),
            "drop_rate": DROP_RATE,
            "img_size": IMG_SIZE,
            "normalize_mean": list(IMAGENET_MEAN),
            "normalize_std": list(IMAGENET_STD),
            "channel_order": "RGB",
            "preprocessing": "ben_graham: circular crop -> resize -> gaussian-subtraction contrast",
            "class_weights": [float(x) for x in class_weights.detach().cpu().tolist()],
            "train_grade_counts": {int(k): int(v) for k, v in counts.to_dict().items()},
            "epoch": epoch,
            "val_qwk": float(best_qwk),
            "seed": SEED,
        }, CKPT_PATH)
        np.save(VAL_LOGITS_PATH, v_logits)
        np.save(VAL_LABELS_PATH, v_labels)
        np.save(VAL_IDS_PATH, v_ids)
        print(f"           saved {os.path.basename(CKPT_PATH)} + val logits {tuple(v_logits.shape)}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nearly stop - no val QWK gain for {PATIENCE} epochs "
                  f"(best epoch {best_epoch}, QWK {best_qwk:.4f})")
            break

print(f"\nbest epoch {best_epoch} | val QWK {best_qwk:.4f}")

## 10. Held-out test evaluation

Reload the best checkpoint, evaluate the untouched test split, print final kappa + referable sensitivity/specificity + the 5-class confusion matrix, and re-save the val/test logits from the restored best weights.

In [ ]:
try:
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"loaded best checkpoint: epoch {ckpt['epoch']}, val QWK {ckpt['val_qwk']:.4f}\n")

vm, v_logits, v_labels, v_ids = evaluate(model, val_loader, criterion)
tm, t_logits, t_labels, t_ids = evaluate(model, test_loader, criterion)

np.save(VAL_LOGITS_PATH, v_logits)
np.save(VAL_LABELS_PATH, v_labels)
np.save(VAL_IDS_PATH, v_ids)
np.save(TEST_LOGITS_PATH, t_logits)
np.save(TEST_LABELS_PATH, t_labels)
np.save(TEST_IDS_PATH, t_ids)

print("=" * 60)
print("HELD-OUT TEST RESULTS (best checkpoint)")
print("=" * 60)
print(f"  quadratic-weighted kappa : {tm['qwk']:.4f}     (target > 0.85)")
print(f"  macro F1                 : {tm['macro_f1']:.4f}")
print(f"  referable-DR sensitivity : {tm['ref_sens']:.4f}     (target > 0.90)")
print(f"  referable-DR specificity : {tm['ref_spec']:.4f}     (target > 0.85)")
print(f"    referable confusion    : TP {tm['ref_tp']}  FN {tm['ref_fn']}  "
      f"TN {tm['ref_tn']}  FP {tm['ref_fp']}")
print("\n  5-class confusion (rows = true 0..4, cols = pred 0..4):")
print(confusion_matrix(t_labels, t_logits.argmax(1), labels=list(range(NUM_CLASSES))))
print(f"\n  (val QWK {vm['qwk']:.4f} | val refDR sens {vm['ref_sens']:.3f} "
      f"spec {vm['ref_spec']:.3f})")

def _j(d):
    return {k: (v if isinstance(v, int) else float(v)) for k, v in d.items()}

with open(METRICS_PATH, "w") as fh:
    json.dump({"val": _j(vm), "test": _j(tm), "best_epoch": best_epoch, "history": history,
               "config": {"model": MODEL_NAME, "img_size": IMG_SIZE, "batch": BATCH_SIZE,
                          "epochs": EPOCHS, "patience": PATIENCE, "lr": LR,
                          "weight_decay": WEIGHT_DECAY, "drop_rate": DROP_RATE, "seed": SEED}},
              fh, indent=2)
print("\nwrote", METRICS_PATH)

## 11. Artifacts to download

After the run, grab these from the Kaggle **Output** panel (right sidebar), or Save Version and download from the version's Output tab.

In [ ]:
print("Download from the Kaggle 'Output' panel after the run:\n")
for p in [CKPT_PATH, VAL_LOGITS_PATH, VAL_LABELS_PATH, VAL_IDS_PATH,
          TEST_LOGITS_PATH, TEST_LABELS_PATH, TEST_IDS_PATH, METRICS_PATH]:
    tag = f"{os.path.getsize(p) / 1e6:8.2f} MB" if os.path.isfile(p) else "  MISSING"
    print(f"  {os.path.basename(p):32s} {tag}")

msg = (
    "\nNext (Prompt 6a, local):\n"
    "  branchA_v1.pt              -> central-system/backend/ml-pipeline/models/branchA_v1.pt\n"
    "  branchA_v1_val_logits.npy  -> models/   (temperature-scaling input)\n"
    "  branchA_v1_val_labels.npy  -> models/\n"
    "  then fit_temperature(val_logits, val_labels) calibrates the softmax.\n"
)
print(msg)